In [ ]:
# @title Email Data Scraper
# Force upgrade to ensure we have the latest table handling
!pip install --upgrade gspread beautifulsoup4 oauth2client --quiet

import imaplib
import email
from email.utils import parsedate_to_datetime
from bs4 import BeautifulSoup
import gspread
from google.colab import auth
from google.auth import default
from gspread.utils import rowcol_to_a1
import time
import re
import html

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================

IMAP_SERVER = 'imap.seznam.cz' # ⚠️ REPLACE THIS
EMAIL_USER = 'abc@abc.com' # ⚠️ REPLACE THIS
EMAIL_PASS = 'PASSWORD' # ⚠️ REPLACE THIS
SHEET_NAME = 'Email Leads' # ⚠️ REPLACE THIS

# 'UNSEEN' = Only new. 'SEEN' = Read emails. 'ALL' = Everything.
SEARCH_CRITERIA = 'SEEN'
BATCH_SIZE = 10

# 🔥 HEADERS (Matches Apps Script)
HEADERS = [
    'Datum',          # A
    'Společnost',     # B
    'Jméno',          # C
    'Příjmení',       # D
    'Telefon',        # E
    'Email',          # F
    'Posláno',        # G
    'Rodné číslo',    # H
    '', '', '',       # I, J, K
    'Částka'          # L
]

# ==========================================
# 🛠️ HELPER FUNCTIONS
# ==========================================

def clean_text(text):
    if not text: return ""
    text = html.unescape(text)
    return text.strip().replace('\xa0', ' ').replace('\r', '').replace('\n', ' ')

def rigorous_decode(payload, charset=None):
    if charset and 'utf-8' not in charset.lower():
        try:
            return payload.decode(charset)
        except:
            pass
    try:
        return payload.decode('utf-8', errors='replace')
    except:
        pass
    try:
        return payload.decode('windows-1250', errors='replace')
    except:
        return payload.decode('latin1', errors='replace')

def format_amount(value):
    """
    Formats '100000 Kč' -> '100.000'
    REMOVED apostrophe. Ensure Sheet Locale is 'Czechia' to avoid 35.000 -> 35 issue.
    """
    if not value: return ""
    num = re.sub(r'\D', '', value)
    if not num: return ""
    # Format with dots: 100000 -> 100.000
    return "{:,}".format(int(num)).replace(",", ".")

def clean_rc(value):
    if not value: return ""
    return re.sub(r'[\/\s]', '', value)

def parse_7finance(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    data = {}

    selectors = {
        'jmeno': 'table:nth-child(4) tr:nth-child(2) strong',
        'prijmeni': 'table:nth-child(4) tr:nth-child(3) strong',
        'telefon': 'tr:nth-child(8) strong',
        'vyse': 'table:nth-child(4) tr:nth-child(1) strong',
        'rc': 'table:nth-child(4) tr:nth-child(4) strong'
    }

    for key, selector in selectors.items():
        elem = soup.select_one(selector)
        if elem: data[key] = clean_text(elem.get_text())

    # Email Logic
    email_elem = soup.select_one('tr:nth-child(4) a')
    if not email_elem:
        email_elem = soup.select_one('tr:nth-child(9) a')

    if email_elem:
        data['email'] = clean_text(email_elem.get_text())
    else:
        target = soup.find(string=re.compile("E-mail|Email", re.IGNORECASE))
        if target:
            parent = target.find_parent(['tr', 'td'])
            if parent:
                clean_e = parent.get_text().replace('E-mail', '').replace('Email', '').replace(':', '')
                data['email'] = clean_text(clean_e)

    if 'jmeno' in data and ' ' in data['jmeno']:
        parts = data['jmeno'].split()
        data['jmeno'] = parts[0]
        if 'prijmeni' not in data or not data['prijmeni']:
            data['prijmeni'] = " ".join(parts[1:])

    return data

def parse_volsor(text_content):
    data = {'jmeno': '', 'prijmeni': '', 'telefon': '', 'email': '', 'vyse': '', 'rc': ''}
    full_name = ""
    lines = text_content.splitlines()
    for line in lines:
        if ':' in line:
            parts = line.split(':', 1)
            key = parts[0].lower().strip()
            val = clean_text(parts[1])

            if 'jméno' in key: full_name = val
            if 'výše půjčky' in key: data['vyse'] = val
            if 'telefon' in key: data['telefon'] = val
            if 'e-mail' in key: data['email'] = val
            if 'rodné číslo' in key: data['rc'] = val

    if full_name:
        if ' ' in full_name:
            split = full_name.split(' ', 1)
            data['jmeno'] = split[0]
            data['prijmeni'] = split[1]
        else:
            data['jmeno'] = full_name

    return data

def get_decoded_body(msg):
    body = ""
    charset = msg.get_content_charset()

    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/html":
                return rigorous_decode(part.get_payload(decode=True), part.get_content_charset()), "html"
            elif part.get_content_type() == "text/plain" and body == "":
                body = rigorous_decode(part.get_payload(decode=True), part.get_content_charset())
        return body, "text"
    else:
        return rigorous_decode(msg.get_payload(decode=True), charset), msg.get_content_type()

def save_safe(worksheet, rows):
    if not rows: return

    all_values = worksheet.get_all_values()
    next_row_idx = len(all_values) + 1

    start_cell = f"A{next_row_idx}"
    end_cell = f"L{next_row_idx + len(rows) - 1}"
    range_name = f"{start_cell}:{end_cell}"

    worksheet.update(values=rows, range_name=range_name)

    try:
        cb_start = f"G{next_row_idx}"
        cb_end = f"G{next_row_idx + len(rows) - 1}"
        validation_rule = gspread.utils.DataValidationRule(
            gspread.utils.BooleanCondition('BOOLEAN'),
            showCustomUi=True
        )
        gspread.utils.set_data_validation_for_cell_range(worksheet, f"{cb_start}:{cb_end}", validation_rule)
    except:
        pass

# ==========================================
# 🚀 MAIN EXECUTION
# ==========================================

def main():
    print("🔐 Authenticating...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    try:
        worksheet = gc.open(SHEET_NAME).sheet1
    except:
        print(f"❌ Sheet '{SHEET_NAME}' not found.")
        return

    current_headers = worksheet.row_values(1)
    if not current_headers:
        print("📝 Adding Headers...")
        worksheet.append_row(HEADERS)
    elif current_headers != HEADERS:
        print("⚠️ Warning: Headers might not match. Check sheet.")

    known_emails = set()
    existing_data = worksheet.get_all_values()
    if len(existing_data) > 1:
        for row in existing_data[1:]:
            if len(row) > 5: known_emails.add(row[5])

    print(f"📧 Connecting to {IMAP_SERVER}...")
    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    try:
        mail.login(EMAIL_USER, EMAIL_PASS)
        mail.select("INBOX")
    except Exception as e:
        print(f"❌ Login Failed: {e}")
        return

    senders = {
        'partner@novyklient.cz': '7Finance',
        'volsor@cdsec24.cz': 'Volsor'
    }

    batch_rows = []
    total_added = 0

    for sender_email, lead_type in senders.items():
        print(f"🔍 Scanning {lead_type}...")
        status, messages = mail.search(None, f'(FROM "{sender_email}" {SEARCH_CRITERIA})')
        email_ids = messages[0].split()
        print(f"   Found {len(email_ids)} emails.")

        for i, e_id in enumerate(email_ids):
            try:
                _, msg_data = mail.fetch(e_id, "(RFC822)")
                msg = email.message_from_bytes(msg_data[0][1])

                date_str = ""
                if msg['Date']:
                    dt = parsedate_to_datetime(msg['Date'])
                    date_str = dt.strftime("%d.%m.%Y")

                body, ctype = get_decoded_body(msg)

                parsed = {}
                if lead_type == '7Finance':
                    if 'html' not in ctype: continue
                    parsed = parse_7finance(body)
                elif lead_type == 'Volsor':
                    text_content = body
                    if 'html' in ctype:
                        soup = BeautifulSoup(body, 'html.parser')
                        text_content = soup.get_text('\n')
                    parsed = parse_volsor(text_content)

                lead_email = parsed.get('email', '')
                if lead_email in known_emails: continue
                if lead_email: known_emails.add(lead_email)

                clean_amount = format_amount(parsed.get('vyse', ''))
                clean_rodne = clean_rc(parsed.get('rc', ''))

                if parsed.get('jmeno') or parsed.get('email'):
                    row = [
                        date_str,
                        lead_type,
                        parsed.get('jmeno', ''),
                        parsed.get('prijmeni', ''),
                        parsed.get('telefon', ''),
                        lead_email,
                        False,
                        clean_rodne,
                        "", "", "",
                        clean_amount
                    ]
                    batch_rows.append(row)
                    total_added += 1

                if len(batch_rows) >= BATCH_SIZE:
                    print(f"   💾 Saving batch of {len(batch_rows)} leads...", end=" ")
                    save_safe(worksheet, batch_rows)
                    print("Done ✅")
                    batch_rows = []
                    time.sleep(1)

            except Exception as e:
                print(f"   ⚠️ Error processing email {e_id}: {e}")
                continue

    if batch_rows:
        print(f"💾 Saving final {len(batch_rows)} leads...", end=" ")
        save_safe(worksheet, batch_rows)
        print("Done ✅")

    mail.close()
    mail.logout()
    print(f"\n🎉 Finished! Added {total_added} new leads.")

if __name__ == "__main__":
    main()

🔐 Authenticating...
📝 Adding Headers...
📧 Connecting to imap.seznam.cz...
🔍 Scanning 7Finance...
   Found 2463 emails.
   💾 Saving batch of 10 leads... Done ✅
   💾 Saving batch of 10 leads... Done ✅
   💾 Saving batch of 10 leads... Done ✅
   💾 Saving batch of 10 leads... Done ✅
   💾 Saving batch of 10 leads... Done ✅
   💾 Saving batch of 10 leads... Done ✅


KeyboardInterrupt: 